# Pontificia Universidad Católica del Perú
## Facultad de Ciencias e Ingeniería
### Tesis: Predicción de tráfico y velocidad de flujo vehicular en Lima Metropolitana mediante técnicas de Deep Learning, empleando datos de vehículos flotantes FCD y grafos viales

**Autor:** Oscar Lau Lu  
**Asesor:** Dr. César Armando Beltrán Castañón  
**Módulo:** Preprocesamiento de Datos de Telemetría Vial (Objetivo Específico 2 - C2)

---
### Objetivos de este Cuaderno:
- **RE 2.1 (Limpieza de datos):** Tratamiento de la información cruda, filtrado de velocidades anómalas e imposibles (Kaya & Utku Kalay, 2025), regularización temporal e imputación de vacíos (Li et al., 2021) y generación de log de auditoría.
- **RE 2.2 (Estudio estadístico):** Análisis exploratorio de variables de tráfico (media, varianza, distribuciones y series temporales en horas punta).
- **RE 2.3 (Dataset unificado):** Generación de dataset tabular limpio y estructuración del tensor espacio-temporal (T, N, F) para modelos Deep Learning (GCN + LSTM/GRU).

### 1. Importación de Librerías y Configuración del Entorno

In [ ]:
import os
import json
import logging
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuración visual de gráficos para reportes de tesis
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

### 2. Configuración del Log de Ejecución (Medio de Verificación RE 2.1)
El marco lógico de la tesis exige como medio de verificación un log de ejecución que documente formalmente el tratamiento de la data cruda.

In [ ]:
# Configuración de rutas del proyecto
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
DATA_PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(os.path.join(RESULTS_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "models"), exist_ok=True)
os.makedirs(DATA_PROC_DIR, exist_ok=True)

LOG_FILE = os.path.join(RESULTS_DIR, "logs", "preprocessing_execution.log")

logger = logging.getLogger("TesisPreprocess")
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()

formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

ch = logging.StreamHandler()
ch.setFormatter(formatter)
logger.addHandler(ch)

fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
fh.setFormatter(formatter)
logger.addHandler(fh)

logger.info("=== Inicio de Sesión de Preprocesamiento de Telemetría ===")


### 3. Carga e Inspección Inicial del Dataset Crudo de TomTom

In [ ]:
raw_candidate = os.path.join(DATA_RAW_DIR, "tomtom_lima_traffic_original.csv")
RAW_DATA_PATH = raw_candidate if os.path.exists(raw_candidate) else "tomtom_lima_traffic_original.csv"

# Lectura de datos
df_raw = pd.read_csv(RAW_DATA_PATH)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])

logger.info(f"Archivo cargado: {RAW_DATA_PATH}")
logger.info(f"Dimensiones iniciales: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas")
logger.info(f"Rango temporal: {df_raw['timestamp'].min()} hasta {df_raw['timestamp'].max()}")
logger.info(f"Nodos/segmentos únicos: {df_raw['segment_id'].nunique()}")

df_raw.head()


In [ ]:
print("--- Información del DataFrame ---")
df_raw.info()
print("\n--- Valores nulos por columna ---")
print(df_raw.isna().sum())

### 4. Fase 1: Filtrado de Anomalías y Duplicados (Kaya & Utku Kalay, 2025)
Se descartan valores imposibles para vías urbanas ($v < 0$ o $v > 130$ km/h, $v_{freeflow} \le 0$) e inconsistencias en los índices de congestión y confianza.

In [ ]:
MIN_SPEED = 0.0
MAX_SPEED = 130.0

initial_count = len(df_raw)

# 1. Eliminación de duplicados exactos
dup_mask = df_raw.duplicated(subset=["timestamp", "segment_id"])
n_dups = int(dup_mask.sum())
df_clean = df_raw[~dup_mask].copy()

# 2. Filtrado de velocidades físicamente anómalas
speed_anomalies = (
    (df_clean["current_speed"] < MIN_SPEED) |
    (df_clean["current_speed"] > MAX_SPEED) |
    (df_clean["freeflow_speed"] <= 0) |
    (df_clean["freeflow_speed"] > MAX_SPEED)
)
n_speed_anomalies = int(speed_anomalies.sum())
df_clean = df_clean[~speed_anomalies].copy()

# 3. Filtrado de índices fuera del rango [0, 1]
index_anomalies = (
    (df_clean["congestion_index"] < 0.0) |
    (df_clean["congestion_index"] > 1.0) |
    (df_clean["confidence"] < 0.0) |
    (df_clean["confidence"] > 1.0)
)
n_index_anomalies = int(index_anomalies.sum())
df_clean = df_clean[~index_anomalies].copy()

logger.info(f"Duplicados eliminados: {n_dups}")
logger.info(f"Anomalías de velocidad descartadas: {n_speed_anomalies}")
logger.info(f"Anomalías de índices descartadas: {n_index_anomalies}")
logger.info(f"Registros restantes tras Fase 1: {len(df_clean)} de {initial_count}")

### 5. Fase 2: Regularización Temporal e Imputación de Faltantes (Li et al., 2021; Geng et al., 2026)
Para alimentar arquitecturas espaciotemporales (GCN + LSTM), cada paso de tiempo debe tener observaciones de **todos los $N=34$ nodos viales** de forma sincrónica.
1. Se redondean los timestamps a intervalos regulares de 15 minutos (`15min`).
2. Se filtran las fechas donde los 34 segmentos completos estuvieron en recolección activa (a partir de septiembre).
3. Se construye la cuadrícula producto $(T \times N)$ y se imputan los vacíos mediante interpolación lineal temporal por tramo.

In [ ]:
FREQ = "15min"
df_clean["timestamp_aligned"] = df_clean["timestamp"].dt.round(FREQ)

# Promediar en caso de colisiones en la misma ventana de 15min
if df_clean.duplicated(subset=["timestamp_aligned", "segment_id"]).any():
    agg_dict = {
        "timestamp": "first",
        "segment_name": "first",
        "zone": "first",
        "latitude": "first",
        "longitude": "first",
        "current_speed": "mean",
        "freeflow_speed": "mean",
        "confidence": "mean",
        "congestion_index": "mean",
    }
    df_clean = df_clean.groupby(["timestamp_aligned", "segment_id"], as_index=False).agg(agg_dict)

# Conservar los días con los 34 segmentos completos del grafo piloto
all_segments = sorted(df_clean["segment_id"].unique())
n_segments = len(all_segments)

daily_counts = df_clean.groupby(df_clean["timestamp_aligned"].dt.date)["segment_id"].nunique()
complete_dates = daily_counts[daily_counts == n_segments].index

logger.info(f"Días con monitoreo completo de {n_segments} nodos: {len(complete_dates)} días")
df_filtered = df_clean[df_clean["timestamp_aligned"].dt.date.isin(complete_dates)].copy()

# Mapa de metadatos estáticos por segmento
meta_cols = ["segment_id", "segment_name", "zone", "latitude", "longitude"]
seg_meta = df_filtered[meta_cols].drop_duplicates(subset=["segment_id"]).set_index("segment_id")

# Producto cartesiano T x N
unique_ts = sorted(df_filtered["timestamp_aligned"].unique())
logger.info(f"Total de instantes temporales T={len(unique_ts)}, Nodos N={n_segments}. Total esperado={len(unique_ts) * n_segments}")

grid = pd.MultiIndex.from_product(
    [unique_ts, all_segments],
    names=["timestamp_aligned", "segment_id"]
).to_frame().reset_index(drop=True)

df_grid = pd.merge(grid, df_filtered, on=["timestamp_aligned", "segment_id"], how="left")
missing_before = int(df_grid["current_speed"].isna().sum())
logger.info(f"Valores faltantes a imputar en la cuadrícula completa: {missing_before} ({missing_before / len(df_grid):.2%})")

# Restaurar metadatos estáticos en registros imputados
df_grid["segment_name"] = df_grid["segment_id"].map(seg_meta["segment_name"])
df_grid["zone"] = df_grid["segment_id"].map(seg_meta["zone"])
df_grid["latitude"] = df_grid["segment_id"].map(seg_meta["latitude"])
df_grid["longitude"] = df_grid["segment_id"].map(seg_meta["longitude"])
df_grid["timestamp"] = df_grid["timestamp_aligned"]

# Imputación temporal lineal por segmento
numeric_cols = ["current_speed", "freeflow_speed", "congestion_index", "confidence"]
df_grid = df_grid.sort_values(by=["segment_id", "timestamp_aligned"]).reset_index(drop=True)

for col in numeric_cols:
    df_grid[col] = df_grid.groupby("segment_id")[col].transform(lambda s: s.interpolate(method="linear").bfill().ffill())
    if df_grid[col].isna().any():
        df_grid[col] = df_grid[col].fillna(df_filtered[col].mean())

logger.info(f"Imputación completada con éxito. Nulos restantes: {df_grid[numeric_cols].isna().sum().sum()}")

### 6. Fase 3: Feature Engineering Cíclico y Temporal (Buroni et al., 2021)
Para capturar la estacionalidad diaria y semanal sin discontinuidades entre extremos (ej. entre 23:59 y 00:00), se proyectan las horas y días a coordenadas circulares:
$$\text{sin\_hour} = \sin\left(\frac{2\pi \cdot t}{24}\right), \quad \text{cos\_hour} = \cos\left(\frac{2\pi \cdot t}{24}\right)$$
$$\text{sin\_dow} = \sin\left(\frac{2\pi \cdot d}{7}\right), \quad \text{cos\_dow} = \cos\left(\frac{2\pi \cdot d}{7}\right)$$

In [ ]:
ts = df_grid["timestamp_aligned"]
hour_float = ts.dt.hour + ts.dt.minute / 60.0
dow = ts.dt.dayofweek

# Componentes cíclicos
df_grid["sin_hour"] = np.sin(2 * np.pi * hour_float / 24.0).round(6)
df_grid["cos_hour"] = np.cos(2 * np.pi * hour_float / 24.0).round(6)
df_grid["sin_dow"] = np.sin(2 * np.pi * dow / 7.0).round(6)
df_grid["cos_dow"] = np.cos(2 * np.pi * dow / 7.0).round(6)

# Variables contextuales de movilidad
df_grid["is_weekend"] = dow.isin([5, 6]).astype(int)
is_morning_rush = (hour_float >= 7.0) & (hour_float <= 9.5)
is_evening_rush = (hour_float >= 17.5) & (hour_float <= 20.5)
df_grid["is_rush_hour"] = (is_morning_rush | is_evening_rush).astype(int)

# Razón de velocidad actual vs flujo libre
df_grid["speed_ratio"] = (df_grid["current_speed"] / df_grid["freeflow_speed"]).clip(0.0, 2.0).round(4)

df_grid[["timestamp_aligned", "segment_id", "current_speed", "sin_hour", "cos_hour", "is_rush_hour", "speed_ratio"]].head()

### 7. Fase 4: Normalización MinMax de Variables Continuas (Kaya & Utku Kalay, 2025)
Se normalizan las variables continuas en el rango $[0, 1]$ para estabilidad del entrenamiento y se guardan los parámetros mínimos y máximos en `scaler_params.json`.

In [ ]:
norm_cols = ["current_speed", "freeflow_speed", "congestion_index"]
scaler_params = {}

for col in norm_cols:
    c_min = float(df_grid[col].min())
    c_max = float(df_grid[col].max())
    denom = c_max - c_min if c_max != c_min else 1.0
    scaler_params[col] = {"min": c_min, "max": c_max}
    df_grid[f"{col}_norm"] = ((df_grid[col] - c_min) / denom).clip(0.0, 1.0).round(6)

SCALER_PATH = os.path.join(RESULTS_DIR, "models", "scaler_params.json")
with open(SCALER_PATH, "w", encoding="utf-8") as f:
    json.dump(scaler_params, f, indent=4)

logger.info(f"Parámetros de normalización exportados a {SCALER_PATH}")
print(json.dumps(scaler_params, indent=2))

### 8. Estudio Estadístico de los Datos Recopilados (RE 2.2)
Generación de medidas estadísticas cuantitativas (media, desviación estándar, medianas, percentiles y extremos) por segmento vial.

In [ ]:
stats_by_segment = df_grid.groupby("segment_id")[["current_speed", "freeflow_speed", "congestion_index"]].agg(
    ["count", "mean", "std", "min", "median", "max"]
)
stats_by_segment.to_csv("traffic_summary_statistics.csv")
print("Resumen estadístico por segmento (primeras 5 vías):")
stats_by_segment.head()

#### 8.1 Visualizaciones de Distribución y Patrones de Tráfico Horario

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Distribución de velocidades
axes[0].hist(df_grid["current_speed"], bins=25, alpha=0.75, label="Velocidad Actual (km/h)", color="#1f77b4", edgecolor="black")
axes[0].hist(df_grid["freeflow_speed"], bins=25, alpha=0.55, label="Flujo Libre (km/h)", color="#2ca02c", edgecolor="black")
axes[0].set_title("Distribución de Velocidad Real vs Flujo Libre")
axes[0].set_xlabel("Velocidad (km/h)")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# 2. Patrón de congestión promedio según la hora del día
hourly_congestion = df_grid.groupby(df_grid["timestamp_aligned"].dt.hour)["congestion_index"].mean()
axes[1].plot(hourly_congestion.index, hourly_congestion.values, marker="o", color="#d62728", linewidth=2.5)
axes[1].axvspan(7, 9.5, color="orange", alpha=0.2, label="Hora Punta Mañana (07:00-09:30)")
axes[1].axvspan(17.5, 20.5, color="red", alpha=0.15, label="Hora Punta Tarde (17:30-20:30)")
axes[1].set_title("Curva Promedio de Congestión Vehicular en Lima")
axes[1].set_xlabel("Hora del Día")
axes[1].set_ylabel("Índice de Congestión [0 - 1]")
axes[1].legend()

plt.tight_layout()
FIGURE_PATH = os.path.join(RESULTS_DIR, "figures", "grafico_patron_congestion.png")
plt.savefig(FIGURE_PATH, dpi=300)
plt.show()

### 9. Exportación del Dataset Limpio Tabular (RE 2.3)
Se exporta el archivo `tomtom_traffic_preprocessed.csv` con consistencia relacional completa.

In [ ]:
OUTPUT_TABULAR_PATH = os.path.join(DATA_PROC_DIR, "tomtom_traffic_preprocessed.csv")

cols_order = [
    "timestamp_aligned", "segment_id", "segment_name", "zone",
    "latitude", "longitude", "current_speed", "freeflow_speed",
    "congestion_index", "confidence", "current_speed_norm",
    "freeflow_speed_norm", "congestion_index_norm", "sin_hour",
    "cos_hour", "sin_dow", "cos_dow", "is_weekend", "is_rush_hour", "speed_ratio"
]

df_grid = df_grid.sort_values(by=["timestamp_aligned", "segment_id"]).reset_index(drop=True)
df_grid[cols_order].to_csv(OUTPUT_TABULAR_PATH, index=False)
logger.info(f"Dataset tabular exportado a {OUTPUT_TABULAR_PATH} con {len(df_grid)} filas.")
print(f"✓ Dataset exportado exitosamente: {OUTPUT_TABULAR_PATH}")

### 10. Construcción del Tensor Espacio-Temporal (T, N, F) para Deep Learning en Grafos
Para los modelos espacio-temporales híbridos (GCN + LSTM/GRU) del Objetivo Específico 3 (C3), se genera un tensor multidimensional tridimensional:
- **T**: Instantes temporales sincronizados
- **N**: 34 nodos viales del grafo piloto
- **F**: Características por nodo (`current_speed_norm`, `freeflow_speed_norm`, `congestion_index_norm`, variables temporales y cíclicas)

In [ ]:
TENSOR_PATH = os.path.join(DATA_PROC_DIR, "traffic_tensor_34nodes.npz")

feature_list = [
    "current_speed_norm",
    "freeflow_speed_norm",
    "congestion_index_norm",
    "sin_hour",
    "cos_hour",
    "sin_dow",
    "cos_dow",
    "is_weekend",
    "is_rush_hour",
    "speed_ratio"
]

timestamps_array = sorted(df_grid["timestamp_aligned"].unique())
segments_array = sorted(df_grid["segment_id"].unique())

T = len(timestamps_array)
N = len(segments_array)
F = len(feature_list)

tensor_data = df_grid[feature_list].values.reshape((T, N, F))

logger.info(f"Dimensiones del tensor generado: (T={T}, N={N}, F={F})")

np.savez_compressed(
    TENSOR_PATH,
    data=tensor_data,
    timestamps=np.array([str(t) for t in timestamps_array]),
    segments=np.array(segments_array),
    features=np.array(feature_list)
)

print(f"✓ Tensor espacio-temporal guardado en: {TENSOR_PATH}")
print(f"Forma del Tensor: {tensor_data.shape} -> (T={T} instantes, N={N} nodos viales, F={F} características)")

### 11. Verificación de Carga Directa en PyTorch

In [ ]:
import torch

# Carga directa del tensor en formato PyTorch Tensor
data = np.load(TENSOR_PATH)
tensor_torch = torch.tensor(data["data"], dtype=torch.float32)

print("Tensor PyTorch creado exitosamente:")
print(f"- Type: {tensor_torch.dtype}")
print(f"- Shape: {tensor_torch.shape}")
print(f"- Dispositivo: {tensor_torch.device}")
print(f"- Rango numérico: min={tensor_torch.min():.4f}, max={tensor_torch.max():.4f}")
logger.info("✓ Preprocesamiento finalizado y validado para entrenamiento de modelos Deep Learning.")